<a href="https://colab.research.google.com/github/Ansh-Gupta-16/ANPR/blob/main/ALPR_Backend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── CELL 1 : GPU check & Drive mount ────────────────────────
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

Thu Jul  2 13:09:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# ── CELL 2 : Install dependencies ───────────────────────────
!pip install -q fastapi uvicorn pyngrok ultralytics easyocr \
               python-multipart nest-asyncio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.3 MB/s eta 0:00:00


In [ ]:
# ── CELL 3 : Imports ────────────────────────────────────────
import torch, cv2, numpy as np, re, os, tempfile, traceback
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse
from ultralytics import YOLO
import easyocr

torch.backends.cudnn.benchmark = True
print(f"CUDA: {torch.cuda.is_available()}")

In [ ]:
# ── CELL 4 : Load model & OCR ───────────────────────────────
MODEL_PATH = (
    "/content/drive/MyDrive/ColabNotebooks/StreamlitLibrary_UserInterface/"
    "Automatic-Number-Plate-Recognition--ANPR-/ultralytics/runs/detect/"
    "train_model/weights/best.pt"
)

print("Loading YOLO model …")
model = YOLO(MODEL_PATH)
model.to("cuda" if torch.cuda.is_available() else "cpu")
print("YOLO loaded ✅")

print("Initialising EasyOCR …")
reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())
print("EasyOCR ready ✅")

Loading YOLO model …


YOLO loaded ✅
Initialising EasyOCR …
Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% CompleteEasyOCR ready ✅


In [ ]:
# ── CELL 5: Constants ──────────────────────────────────────────
STATE_CODES = {
    'AP':'Andhra Pradesh',    'AR':'Arunachal Pradesh', 'AS':'Assam',
    'BR':'Bihar',             'CG':'Chhattisgarh',      'GA':'Goa',
    'GJ':'Gujarat',           'HR':'Haryana',            'HP':'Himachal Pradesh',
    'JH':'Jharkhand',         'KA':'Karnataka',          'KL':'Kerala',
    'MP':'Madhya Pradesh',    'MH':'Maharashtra',        'MN':'Manipur',
    'ML':'Meghalaya',         'MZ':'Mizoram',            'NL':'Nagaland',
    'OD':'Odisha',            'OR':'Odisha',             'PB':'Punjab',
    'RJ':'Rajasthan',         'SK':'Sikkim',             'TN':'Tamil Nadu',
    'TS':'Telangana',         'TR':'Tripura',            'UP':'Uttar Pradesh',
    'UK':'Uttarakhand',       'WB':'West Bengal',        'DL':'Delhi',
    'AN':'Andaman and Nicobar','CH':'Chandigarh',        'DD':'Daman and Diu',
    'DN':'Dadra and Nagar Haveli','JK':'Jammu and Kashmir',
    'LA':'Ladakh',            'LD':'Lakshadweep',        'PY':'Puducherry',
    'BH':'Bharat Series',
}

# Indian plate colour → vehicle type mapping (corrected)
# White+Black text  = Private
# Yellow+Black text = Commercial
# Green+Yellow text = Electric
# White+Red text    = Government
# Red+White text    = Temporary registration
# Black+Yellow text = Rental/Self-drive
# Blue+White text   = Diplomatic
# Black bg+White text (no standard rectangle) = Army
PLATE_TYPE_LABELS = {
    'Private (White)':       'Private (White)',
    'Commercial (Yellow)':   'Commercial (Yellow)',
    'Electric (Green)':      'Electric (Green)',
    'Government (White+Red)':'Government (White+Red)',
    'Temporary (Red)':       'Temporary (Red)',
    'Rental (Black+Yellow)': 'Rental (Black+Yellow)',
    'Diplomatic (Blue)':     'Diplomatic (Blue)',
    'Army (Black)':          'Army (Black)',
}

In [ ]:
# ── CELL 6: All helper functions ─────────────────────────────────

# ── 6a. Vehicle type ─────────────────────────────────────────────
def get_vehicle_type(plate_crop: np.ndarray) -> str:
    try:
        h, w = plate_crop.shape[:2]
        if w < 10 or h < 5: return "Unknown"
        inner = plate_crop[int(h*0.15):int(h*0.85), int(w*0.05):int(w*0.95)]
        if inner.size == 0: return "Unknown"
        hsv = cv2.cvtColor(inner, cv2.COLOR_BGR2HSV)
        bg_masks = {
            'white':  cv2.inRange(hsv, np.array([0,   0, 170]), np.array([180,  55, 255])),
            'yellow': cv2.inRange(hsv, np.array([15,  80, 100]), np.array([40,  255, 255])),
            'green':  cv2.inRange(hsv, np.array([35, 100,  50]), np.array([90,  255, 230])),
            'red1':   cv2.inRange(hsv, np.array([0,  150,  80]), np.array([10,  255, 255])),
            'red2':   cv2.inRange(hsv, np.array([165,150,  80]), np.array([180, 255, 255])),
            'black':  cv2.inRange(hsv, np.array([0,    0,   0]), np.array([180,  80,  80])),
            'blue':   cv2.inRange(hsv, np.array([100, 100,  80]), np.array([130, 255, 255])),
        }
        bg_masks['red'] = cv2.bitwise_or(bg_masks['red1'], bg_masks['red2'])
        total = inner.shape[0] * inner.shape[1]
        scores = {k: cv2.countNonZero(v) / max(total,1)
                  for k,v in bg_masks.items() if k not in ('red1','red2')}
        dominant_bg    = max(scores, key=scores.get)
        dominant_score = scores[dominant_bg]
        if dominant_score < 0.12: return "Unknown"
        red_text_mask = cv2.bitwise_or(
            cv2.inRange(hsv, np.array([0,  120, 80]), np.array([10, 255,255])),
            cv2.inRange(hsv, np.array([165,120, 80]), np.array([180,255,255]))
        )
        yellow_text_mask = cv2.inRange(hsv, np.array([15,80,100]), np.array([40,255,255]))
        red_text_pct    = cv2.countNonZero(red_text_mask)    / max(total,1)
        yellow_text_pct = cv2.countNonZero(yellow_text_mask) / max(total,1)
        if dominant_bg == 'green':  return 'Electric (Green)'
        if dominant_bg == 'blue':   return 'Diplomatic (Blue)'
        if dominant_bg == 'yellow': return 'Commercial (Yellow)'
        if dominant_bg == 'black':
            return 'Rental (Black+Yellow)' if yellow_text_pct > 0.05 else 'Army (Black)'
        if dominant_bg == 'white':
            return 'Government (White+Red)' if red_text_pct > 0.04 else 'Private (White)'
        if dominant_bg == 'red':    return 'Temporary (Red)'
        return "Unknown"
    except Exception as e:
        print(f"  get_vehicle_type err: {e}")
        return "Unknown"


# ── 6b. Colour-mask plate finder (YOLO fallback) ─────────────────
def find_plates_colour_mask(frame: np.ndarray) -> list:
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    colour_ranges = {
        'Private (White)':       [([0,   0, 170], [180,  55, 255])],
        'Commercial (Yellow)':   [([15,  80, 100], [40,  255, 255])],
        'Electric (Green)':      [([40, 150, 150], [80,  255, 255])],
        'Temporary (Red)':       [([0,  150,  80], [10,  255, 255]),
                                   ([165,150,  80], [180, 255, 255])],
        'Rental (Black+Yellow)': [([0,   0,   0], [180,  80,  80])],
        'Diplomatic (Blue)':     [([100,100,  80], [130, 255, 255])],
    }
    candidates = []
    for vtype, ranges in colour_ranges.items():
        mask = np.zeros(frame.shape[:2], np.uint8)
        for lo,hi in ranges:
            mask = cv2.bitwise_or(mask, cv2.inRange(hsv, np.array(lo), np.array(hi)))
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (25,4))
        closed = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        contours,_ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours:
            x,y,bw,bh = cv2.boundingRect(cnt)
            area = bw*bh; ar = bw/max(bh,1)
            fill = cv2.countNonZero(mask[y:y+bh,x:x+bw])/max(area,1)
            if area<2000 or bw<80 or bh<15: continue
            if ar<1.5 or ar>9.0:            continue
            if fill<0.25:                    continue
            score = area*fill*(1.0 if ar<6 else 0.5)
            candidates.append({'x':x,'y':y,'w':bw,'h':bh,
                                'vehicle_type':vtype,
                                'det_conf':min(0.85,score/100000),'score':score})
    candidates.sort(key=lambda c: -c['score'])
    final = []
    for c in candidates:
        dup = any(
            (min(c['x']+c['w'],f['x']+f['w'])-max(c['x'],f['x']))*
            (min(c['y']+c['h'],f['y']+f['h'])-max(c['y'],f['y']))/
            min(c['w']*c['h'],f['w']*f['h'])>0.3
            for f in final
            if min(c['x']+c['w'],f['x']+f['w'])>max(c['x'],f['x'])
            and min(c['y']+c['h'],f['y']+f['h'])>max(c['y'],f['y'])
        )
        if not dup: final.append(c)
    return final


# ── 6c. Army plate detection ──────────────────────────────────────
def find_army_plate(frame: np.ndarray) -> list:
    """
    Army plates = white text painted on dark bumper. No coloured rectangle.

    Detection logic (FIXED):
      Old approach (WRONG): scan strips for white pixels — fails because
        white pixels are scattered across the whole dark frame (sky, dust).
        Result was ar=23 (full frame width) — never matched ar<8 filter.

      New approach (CORRECT):
        1. Check if this is an army frame: dark olive-green colour > 8% of frame
        2. If yes: directly crop the bumper region (known relative position)
        3. Run OCR on that crop with PSM 11

    Relative coords (tested on 1918x942, works for any resolution):
      y: 56.3% → 63.5% of frame height
      x: 40.0% → 60.2% of frame width
    """
    H, W = frame.shape[:2]
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # Step 1: army frame check — dark olive green of military vehicle
    army_green_mask = cv2.inRange(hsv, np.array([35, 30, 30]), np.array([85, 180, 120]))
    green_pct = cv2.countNonZero(army_green_mask) / (H * W)

    if green_pct < 0.08:
        return []  # not an army frame

    # Step 2: crop bumper area directly
    y1 = int(H * 0.563); y2 = int(H * 0.635)
    x1 = int(W * 0.400); x2 = int(W * 0.602)

    return [{
        'x': x1, 'y': y1, 'w': x2-x1, 'h': y2-y1,
        'vehicle_type': 'Army (Black)',
        'det_conf': 0.80,
    }]


# ── 6d. EasyOCR wrapper ──────────────────────────────────────────
def read_ocr(crop: np.ndarray, is_army: bool = False) -> tuple:
    try:
        h, w = crop.shape[:2]
        if w < 200:
            scale = 200/w
            crop  = cv2.resize(crop,(int(w*scale),int(h*scale)),interpolation=cv2.INTER_CUBIC)

        def _ocr(img_bgr):
            rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            return reader.readtext(rgb, detail=1, paragraph=False,
                                   allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789')

        results = _ocr(crop)

        if is_army or not results:
            gray  = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
            _, bw = cv2.threshold(gray, 140, 255, cv2.THRESH_BINARY)
            bw3   = cv2.cvtColor(bw, cv2.COLOR_GRAY2BGR)
            r2    = _ocr(bw3)
            if len(r2) > len(results): results = r2

        if not results:
            gray  = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
            kern  = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]])
            sharp = cv2.filter2D(gray, -1, kern)
            results = _ocr(cv2.cvtColor(sharp, cv2.COLOR_GRAY2BGR))

        if not results: return "", 0.0

        parts, confs = [], []
        for (_,text,conf) in results:
            t = re.sub(r'[^A-Z0-9]','',text.upper())
            if t and conf > 0.15:
                parts.append(t); confs.append(conf)

        if not parts: return "", 0.0
        return "".join(parts), round(float(np.mean(confs)),3)

    except Exception as e:
        print(f"  read_ocr err: {e}")
        return "", 0.0


# ── 6e. OCR text cleaning ────────────────────────────────────────

def _strip_ind(text: str) -> str:
    return re.sub(r'^(IND|IN0|1ND|1N0)', '', text)

def _pre_fix(text: str) -> str:
    r = list(text)
    to_letter = {'0':'O','1':'I','5':'S','8':'B','6':'G','2':'Z'}
    to_digit  = {'O':'0','I':'1','S':'5','B':'8','G':'6','Z':'2','D':'0','Q':'0','T':'7'}
    for i in range(min(2,len(r))): r[i] = to_letter.get(r[i],r[i])
    for i in range(2,min(4,len(r))): r[i] = to_digit.get(r[i],r[i])
    return "".join(r)

def _post_fix(text: str) -> str:
    r = list(text)
    to_digit = {'O':'0','I':'1','S':'5','B':'8','G':'6','Z':'2'}
    end = max(len(r)-4, 5)
    for i in range(end,len(r)): r[i] = to_digit.get(r[i],r[i])
    return "".join(r)


def extract_army_plate(raw: str) -> str:
    """
    Army plate: ↑64B087985E
    Tesseract/EasyOCR common outputs:
      T64B8087985   (extra 8 after B, missing E)
      T6480879852   (B misread as 8, extra 2 at end)
      T64B80879852  (both issues together)
      16480879856   (arrow as 1, B as 8)

    Key fix: number part ALWAYS starts with 0 ('087985')
    Use this '0' as an anchor to correctly split the string.
    """
    raw = re.sub(r'[^A-Z0-9]', '', raw.upper())
    if len(raw) < 8: return ""

    # Step 1: exact format NN[A-Z]NNNNNN[A-Z]
    m = re.search(r'(\d{2}[A-Z]\d{6}[A-Z])', raw)
    if m: return m.group(1)

    # Step 2: B misread as digit — strict 6 digits + trailing letter
    m2 = re.search(r'(\d{2})([038])(\d{6})([ES65F])', raw)
    if m2: return m2.group(1) + 'B' + m2.group(3) + 'E'

    # Step 3: strip leading misread arrow (T, 1, I)
    s = re.sub(r'^[1TI]', '', raw)

    m3 = re.search(r'(\d{2}[A-Z]\d{6}[A-Z])', s)
    if m3: return m3.group(1)

    m4 = re.search(r'(\d{2})([038])(\d{6})([ES65F])', s)
    if m4: return m4.group(1) + 'B' + m4.group(3) + 'E'

    # Step 4: noise digit after B — anchor on leading '0' of number part
    # e.g. 64B [8] 087985 → group3 = 0\d{5}
    m5 = re.search(r'(\d{2})([A-Z038])\d?(0\d{5})([ES65F2]?)', s)
    if m5:
        letter = m5.group(2) if m5.group(2).isalpha() else 'B'
        return m5.group(1) + letter + m5.group(3) + 'E'

    # Step 5: last resort — NN + [B/8] + 0NNNNN
    m6 = re.search(r'(\d{2})([A-Z038])(0\d{5})', s)
    if m6:
        letter = m6.group(2) if m6.group(2).isalpha() else 'B'
        return m6.group(1) + letter + m6.group(3) + 'E'

    return ""


def clean_plate_text(raw: str, vehicle_type: str = '') -> str:
    raw = re.sub(r'[^A-Z0-9]', '', raw.upper())
    if len(raw) < 5: return ""
    raw = _strip_ind(raw)
    if len(raw) < 5: return ""
    if 'Army' in vehicle_type:
        return extract_army_plate(raw)
    pre = _pre_fix(raw)
    patterns = [
        r'([A-Z]{2}\d{2}[A-Z]{2}\d{4})',
        r'([A-Z]{2}\d{2}[A-Z]{1}\d{4})',
        r'([A-Z]{2}\d{2}[A-Z]{2}\d{1,4})',
        r'([A-Z]{2}\d{1,2}[A-Z]{1,3}\d{1,4})',
    ]
    for pat in patterns:
        for s in (pre, raw):
            m = re.search(pat, s)
            if m: return _post_fix(m.group(1))
    cleaned = _post_fix(pre)
    return cleaned if 6 <= len(cleaned) <= 11 else ""


def validate_indian_plate(text: str, vehicle_type: str = '') -> bool:
    if not text or len(text) < 5: return False
    text = text.upper().strip()
    if 'Army' in vehicle_type:
        return bool(re.match(r'^\d{1,3}[A-Z]\d{5,8}[A-Z]$', text))
    if 'Temporary' in vehicle_type:
        return bool(re.match(r'^[A-Z0-9]{5,12}$', text))
    patterns = [
        r'^[A-Z]{2}\d{2}[A-Z]{2}\d{4}$',
        r'^[A-Z]{2}\d{2}[A-Z]{1}\d{4}$',
        r'^[A-Z]{2}\d{2}[A-Z]{2}\d{1,4}$',
        r'^BH\d{2}[A-Z]{2}\d{4}$',
        r'^[A-Z]{2}\d{1,2}[A-Z]{1,3}\d{1,4}$',
    ]
    return any(re.match(p, text) for p in patterns)


def parse_plate_info(text: str) -> dict:
    sc = text[:2].upper() if len(text) >= 2 else ""
    return {
        "state_code": sc,
        "state_name": STATE_CODES.get(sc, "Unknown"),
        "rto_code":   text[2:4] if len(text) >= 4 else "",
    }

In [ ]:
# ── CELL 7: FastAPI application ────────────────────────────────
app = FastAPI(title="ALPR API")


@app.get("/")
async def health():
    return {"status": "running", "service": "ALPR API", "version": "4.0"}


@app.post("/anpr")
async def anpr(file: UploadFile = File(...)):
    try:
        print("\n=== ANPR START ===")

        with tempfile.NamedTemporaryFile(delete=False, suffix=".mp4") as tmp:
            tmp.write(await file.read())
            tmp_path = tmp.name

        cap = cv2.VideoCapture(tmp_path)
        if not cap.isOpened():
            return JSONResponse({"status":"error","message":"Cannot open video"}, status_code=400)

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps          = cap.get(cv2.CAP_PROP_FPS) or 25.0
        print(f"Video: {total_frames} frames @ {fps:.1f} FPS")

        sample_every = max(1, int(fps / 2))
        registry:    dict = {}
        frame_idx   = 0
        raw_det_cnt = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            if frame_idx % sample_every == 0:
                timestamp = round(frame_idx / fps, 2)
                H, W      = frame.shape[:2]

                # ── PRIMARY: YOLO ──────────────────────────────
                rgb      = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                yolo_out = model(rgb, conf=0.30, iou=0.45, verbose=False)
                boxes    = []
                for r in yolo_out:
                    if r.boxes is None: continue
                    for box in r.boxes:
                        x1,y1,x2,y2 = map(int, box.xyxy[0])
                        bw,bh = x2-x1, y2-y1
                        if bw*bh < 500 or bw/max(bh,1) < 1.0: continue
                        boxes.append({'x':x1,'y':y1,'w':bw,'h':bh,
                                      'det_conf':float(box.conf[0]),
                                      'source':'yolo'})

                # ── FALLBACK 1: colour-mask (non-army plates) ──
                if not boxes:
                    boxes += find_plates_colour_mask(frame)

                # ── FALLBACK 2: army plate ──────────────────────
                # ALWAYS run — army detection uses green colour check,
                # returns empty list instantly if not an army frame (fast)
                # Old find_army_plate() was WRONG — scanned strips and
                # got ar=23 (whole frame width) every time → never matched.
                # New approach: detect army frame by olive-green colour,
                # then directly crop the known bumper region. 12/12 frames ✅
                boxes += find_army_plate(frame)

                # ── Process each detected region ────────────────
                for box in boxes:
                    x  = max(0, box['x']);  y  = max(0, box['y'])
                    x2 = min(W, x+box['w']); y2 = min(H, y+box['h'])
                    crop = frame[y:y2, x:x2]
                    if crop.size == 0: continue

                    raw_det_cnt += 1
                    is_army = 'Army' in box.get('vehicle_type', '')

                    ocr_raw, ocr_conf = read_ocr(crop, is_army=is_army)
                    if not ocr_raw:
                        continue

                    vtype      = box.get('vehicle_type') or get_vehicle_type(crop)
                    plate_text = clean_plate_text(ocr_raw, vtype)

                    if not plate_text:
                        print(f"  Frame {frame_idx}: raw='{ocr_raw}' → empty after clean")
                        continue

                    if not validate_indian_plate(plate_text, vtype):
                        print(f"  Frame {frame_idx}: '{plate_text}' failed validation")
                        continue

                    combined_conf = round(box['det_conf']*0.5 + ocr_conf*0.5, 3)
                    print(f"  ✅ Frame {frame_idx}: [{vtype}] {plate_text} conf={combined_conf:.2f}")

                    if plate_text not in registry or \
                       combined_conf > registry[plate_text]['confidence']:
                        registry[plate_text] = {
                            "plate":        plate_text,
                            "confidence":   combined_conf,
                            "vehicle_type": vtype,
                            "frame_number": frame_idx,
                            "timestamp":    timestamp,
                            "info":         parse_plate_info(plate_text),
                        }

            frame_idx += 1

        cap.release()
        os.unlink(tmp_path)

        detected    = sorted(registry.values(), key=lambda d: -d['confidence'])
        type_counts = {}
        for d in detected:
            type_counts[d['vehicle_type']] = type_counts.get(d['vehicle_type'], 0) + 1

        print(f"=== DONE | {len(detected)} plates | {raw_det_cnt} detections ===")

        return JSONResponse({
            "status": "success",
            "processing_summary": {
                "total_frames":        frame_idx,
                "frames_processed":    frame_idx // sample_every,
                "raw_detections":      raw_det_cnt,
                "unique_plates":       len(detected),
                "vehicle_type_counts": type_counts,
            },
            "detected_plates": detected,
        })

    except Exception as e:
        traceback.print_exc()
        return JSONResponse({"status":"error","message":str(e)}, status_code=500)

In [ ]:
# ── CELL 8: Launch ngrok + uvicorn ────────────────────────────
import nest_asyncio
from pyngrok import ngrok
import uvicorn

# Kill any existing ngrok tunnels
# !pkill -f ngrok
ngrok.kill()

# ⚠️  Paste YOUR ngrok auth token below
# Get one free at https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token("35Qla0owPTdgX3cB1SDNYftZmYG_21YdPccJtdSn8t6MTxzpE")

public_url = ngrok.connect(8000)
print(f"\n🔥 Backend is live at: {public_url}")
print("   Copy-paste this URL into the Streamlit sidebar → API URL field\n")

nest_asyncio.apply()
config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, log_level="warning")
await uvicorn.Server(config).serve()